# LOO evaluation harness

generate test sets and compute Recall@K / MRR / NDCG

In [4]:
import pandas as pd
import numpy as np

VISITS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step1_outputs\user_visit_events.parquet"
BUSINESS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step1_outputs\business_lookup.parquet"
MOBILITY_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step2_outputs\user_mobility_table.csv"
HUBS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step2_outputs\user_hubs.csv"

visits = pd.read_csv(VISITS_PATH)
businesses = pd.read_csv(BUSINESS_PATH)
mobility = pd.read_csv(MOBILITY_PATH)
hubs = pd.read_csv(HUBS_PATH)

print(visits.shape, businesses.shape, mobility.shape, hubs.shape)
display(visits.head())

(27732, 14) (14027, 10) (1159, 19) (1423, 6)


,user_id,business_id,event_time,event_type,stars_x,name,latitude,longitude,city,state,categories,stars_y,review_count,is_open
0,---zemaUC8WeJeWKqS6p9Q,eX7o_-s5TmDT-DMfTV4cmw,2021-06-23 08:17:42+00:00,review,5.0,Luckys Steakhouse,34.421239,-119.641027,Montecito,CA,"Restaurants, Seafood, Nightlife, Steakhouses, ...",4.0,387,1
1,--z4j_95J4XLJJFrVHgBiw,IP_E4SHLClorhNg4o5bw9Q,2018-10-16 01:36:14+00:00,review,5.0,Rosy's Taco Bar,39.950896,-75.178367,Philadelphia,PA,"Tacos, Nightlife, Bars, Mexican, Restaurants",4.0,282,1
2,-0A675r8EE26FmhYx68KHw,3OGzmGqWwsyGLkhnxrA9Pw,2013-08-24 00:23:11+00:00,review,1.0,Gourmet Pizza Company,27.937948,-82.484684,Tampa,FL,"Pizza, Restaurants, Gluten-Free, Vegan",4.0,185,1
3,-0MIp6WKJ8QvGnYZQ5ETyg,UTgUhbpM5hRVDB8Nu7-lpw,2018-06-11 22:57:47+00:00,review,4.0,China Cafe,39.610539,-75.074803,Franklinville,NJ,"Chinese, Restaurants",4.0,18,1
4,-0MIp6WKJ8QvGnYZQ5ETyg,jwpZzqoPFLuHE-FwLp41cQ,2020-03-15 00:33:43+00:00,review,4.0,Bistro Di Marino,39.918191,-75.075328,Collingswood,NJ,"Italian, Salad, Restaurants, Seafood",4.0,285,1


In [5]:
print(visits.columns.tolist())

['user_id', 'business_id', 'event_time', 'event_type', 'stars_x', 'name', 'latitude', 'longitude', 'city', 'state', 'categories', 'stars_y', 'review_count', 'is_open']


In [6]:
TIME_COL = "event_time"   # replace with your real column name

visits[TIME_COL] = pd.to_datetime(visits[TIME_COL], errors="coerce")
print(visits[TIME_COL].isna().sum())

0


# Building leave-one-out split

In [7]:
REQ_COLS = ["user_id", "business_id", TIME_COL]
missing = [c for c in REQ_COLS if c not in visits.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

visits_loo = visits.dropna(subset=[TIME_COL]).copy()
visits_loo = visits_loo.sort_values(["user_id", TIME_COL])

loo_rows = []
train_rows = []

for uid, dfu in visits_loo.groupby("user_id"):
    if len(dfu) < 5:
        continue

    dfu = dfu.sort_values(TIME_COL)
    train = dfu.iloc[:-1].copy()
    test = dfu.iloc[-1].copy()

    loo_rows.append({
        "user_id": uid,
        "test_business_id": test["business_id"],
        "test_time": test[TIME_COL],
        "n_train_events": len(train),
        "n_total_events": len(dfu),
    })

    train_rows.append(train)

loo_eval = pd.DataFrame(loo_rows)
train_visits = pd.concat(train_rows, ignore_index=True)

print("loo_eval shape:", loo_eval.shape)
print("train_visits shape:", train_visits.shape)
display(loo_eval.head())

loo_eval shape: (3506, 5)
train_visits shape: (20339, 14)


,user_id,test_business_id,test_time,n_train_events,n_total_events
0,-0MIp6WKJ8QvGnYZQ5ETyg,sEwBpFLYsEPu3aUVkh5gwQ,2021-11-27 23:13:27+00:00,18,19
1,-1VbC_BSr9ai4drsuCCJxA,DcBLYSvOuWcNReolRVr12A,2012-03-19 22:20:11+00:00,6,7
2,-3kRxxeqZgFhn1ade6VGQA,VQcCL9PiNL_wkGf-uF3fjg,2016-09-12 21:02:09+00:00,1,2
3,-4x9GJWGHzsjcxjpJSfp7w,EUu9FqaXxIwhRxLgEJ6WiQ,2017-07-01 00:31:31+00:00,2,3
4,-6DoXmdXEy_P5N-QZzntgA,Ifw5wqcChnL4zBigtR7NKA,2015-10-06 18:33:48+00:00,11,12


In [22]:
profiled_users = set(mobility["user_id"])

loo_eval_filtered = loo_eval[loo_eval["user_id"].isin(profiled_users)].copy()
train_visits_filtered = train_visits[train_visits["user_id"].isin(profiled_users)].copy()

print("loo_eval_filtered rows:", len(loo_eval_filtered))
print("loo_eval_filtered unique users:", loo_eval_filtered["user_id"].nunique())
print("train_visits_filtered unique users:", train_visits_filtered["user_id"].nunique())

loo_eval_filtered rows: 1159
loo_eval_filtered unique users: 1159
train_visits_filtered unique users: 1159


# Building train-only visited sets

In [23]:
train_user_visited = (
    train_visits_filtered.groupby("user_id")["business_id"]
    .apply(set)
    .to_dict()
)

print("users with filtered train visited sets:", len(train_user_visited))

users with filtered train visited sets: 1159


In [24]:
biz_pop_train = (
    train_visits_filtered.groupby("business_id")
    .size()
    .reset_index(name="visit_count")
)

businesses_eval = businesses.drop_duplicates(subset=["business_id"]).copy()

if "visit_count" in businesses_eval.columns:
    businesses_eval = businesses_eval.drop(columns=["visit_count"])

businesses_eval = businesses_eval.merge(biz_pop_train, on="business_id", how="left")
businesses_eval["visit_count"] = businesses_eval["visit_count"].fillna(0)

display(businesses_eval.head())

,business_id,name,latitude,longitude,city,state,categories,stars,review_count,is_open,visit_count
0,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,39.953949,-75.143226,Philadelphia,PA,"Sushi Bars, Restaurants, Japanese",4.0,245,1,1.0
1,ROeacJQwBeh05Rqg7F6TCg,BAP,39.943223,-75.162568,Philadelphia,PA,"Korean, Restaurants",4.5,205,1,1.0
2,9OG5YkX1g2GReZM0AskizA,Romano's Macaroni Grill,39.476117,-119.789339,Reno,NV,"Restaurants, Italian",2.5,339,1,1.0
3,QdN72BWoyFypdGJhhI5r7g,Bar One,39.939825,-75.157447,Philadelphia,PA,"Cocktail Bars, Bars, Italian, Nightlife, Resta...",4.0,65,0,2.0
4,kV_Q1oqis8Qli8dUoGpTyQ,Ardmore Pizza,40.006707,-75.289671,Ardmore,PA,"Pizza, Restaurants",3.5,109,1,3.0


In [25]:
from retrieval_utils import retrieve_candidates_for_user, get_user_radius

uid = loo_eval["user_id"].iloc[0]

test_cands = retrieve_candidates_for_user(
    uid,
    businesses_eval,
    mobility,
    hubs,
    train_user_visited,
    top_n=100
)

print("user:", uid)
display(test_cands.head(10))

user: -0MIp6WKJ8QvGnYZQ5ETyg


,business_id,name,latitude,longitude,city,state,categories,stars,review_count,is_open,visit_count,dist_hub_1,dist_hub_2,dist_hub_3,distance_km,pop_score,score,retrieval_mode,radius_km
1633,Ob01LpWh81jAtzpYirBxpA,Bangkok City,39.853118,-74.982355,Voorhees,NJ,"Restaurants, Thai",4.0,130,1,3.0,0.622673,15.266558,15.419801,0.622673,1.386294,1.324027,hub_aware,5.0
11647,1yr6foIrWy0HBXVpD5Wb3w,Tandoori Kabab,39.854305,-75.006979,Voorhees,NJ,"Restaurants, Indian, Bangladeshi, Pakistani",4.0,83,1,3.0,1.824246,15.259933,14.336468,1.824246,1.386294,1.203870,hub_aware,5.0
6570,O-6k7VRRnwxwve_Zyb-a1g,Akira Japanese Restaurant,39.852832,-74.982520,Voorhees,NJ,"Sushi Bars, Japanese, Restaurants, Asian Fusion",3.5,111,1,2.0,0.589060,15.233125,15.386068,0.589060,1.098612,1.039706,hub_aware,5.0
1875,qjSU3SL8CaGfmOL_BgrqoA,Ritz 16,39.846822,-74.979283,Voorhees,NJ,"Restaurants, Cinema, American (New), Arts & En...",3.0,14,0,2.0,0.682492,14.612386,15.042829,0.682492,1.098612,1.030363,hub_aware,5.0
1028,7hsCR9k_GND2QoQyVkITBg,Pho Voorhees,39.847977,-74.977224,Voorhees,NJ,"Bubble Tea, Soup, Restaurants, Vietnamese, Food",4.5,132,1,2.0,0.828694,14.767908,15.254653,0.828694,1.098612,1.015743,hub_aware,5.0
10021,JssqRnzEuVDuxgbrks87Pg,Tiffin,39.842229,-74.995927,Voorhees,NJ,"Vegetarian, Indian, Halal, Restaurants",4.0,48,0,2.0,1.057560,13.945700,13.753440,1.057560,1.098612,0.992856,hub_aware,5.0
12390,P3jyuANUyuwc9P1w6J_IKQ,Elena Wu Restaurant & Sushi Bar,39.851312,-74.999448,Voorhees,NJ,"Asian Fusion, Restaurants, Chinese, Sushi Bars...",3.5,90,1,2.0,1.110523,14.938815,14.400444,1.110523,1.098612,0.987560,hub_aware,5.0
2374,Nkcu2EdGt34fhUHyKEjXMQ,Lai Thai Cuisine,39.841656,-74.996635,Voorhees,NJ,"Restaurants, Thai",4.0,105,1,2.0,1.145218,13.878357,13.666157,1.145218,1.098612,0.984090,hub_aware,5.0
4875,lML_PZePqlBBWpD1tqRdaA,Nimit Palace,39.840816,-74.996743,Voorhees,NJ,"Vegetarian, Chinese, Food, Seafood, Indian, Pa...",4.5,370,1,2.0,1.217630,13.784567,13.586277,1.217630,1.098612,0.976849,hub_aware,5.0
13549,grVoVN3n0M7IV1jRUiHWHQ,BurgerBarr,39.739750,-75.077300,Sewell,NJ,"Burgers, American (Traditional), American (New...",4.5,217,1,2.0,14.370604,6.568263,1.440948,1.440948,1.098612,0.954517,hub_aware,5.0


# Testing it on users

In [26]:
import math

def recall_at_k(rank, k):
    if rank is None:
        return 0.0
    return 1.0 if rank <= k else 0.0

def mrr(rank):
    if rank is None:
        return 0.0
    return 1.0 / rank

def ndcg_at_k(rank, k):
    if rank is None or rank > k:
        return 0.0
    return 1.0 / math.log2(rank + 1)

In [27]:
def evaluate_one_user(uid, test_business_id, top_n=100):
    cands = retrieve_candidates_for_user(
        uid,
        businesses_eval,
        mobility,
        hubs,
        train_user_visited,
        top_n=top_n
    ).copy()

    if len(cands) == 0:
        return {
            "user_id": uid,
            "test_business_id": test_business_id,
            "found": 0,
            "rank": None,
            "recall@10": 0.0,
            "recall@20": 0.0,
            "mrr": 0.0,
            "ndcg@10": 0.0,
            "ndcg@20": 0.0,
        }

    cands = cands.reset_index(drop=True)
    cands["rank"] = np.arange(1, len(cands) + 1)

    match = cands[cands["business_id"] == test_business_id]

    if len(match) == 0:
        rank = None
        found = 0
    else:
        rank = int(match["rank"].iloc[0])
        found = 1

    return {
        "user_id": uid,
        "test_business_id": test_business_id,
        "found": found,
        "rank": rank,
        "recall@10": recall_at_k(rank, 10),
        "recall@20": recall_at_k(rank, 20),
        "mrr": mrr(rank),
        "ndcg@10": ndcg_at_k(rank, 10),
        "ndcg@20": ndcg_at_k(rank, 20),
    }

In [28]:
eval_rows = []

for _, row in loo_eval_filtered.iterrows():
    out = evaluate_one_user(
        uid=row["user_id"],
        test_business_id=row["test_business_id"],
        top_n=100
    )
    eval_rows.append(out)

loo_results = pd.DataFrame(eval_rows)

print("loo_results rows:", len(loo_results))
print("loo_results unique users:", loo_results["user_id"].nunique())
display(loo_results.head())

loo_results rows: 1159
loo_results unique users: 1159


,user_id,test_business_id,found,rank,recall@10,recall@20,mrr,ndcg@10,ndcg@20
0,-0MIp6WKJ8QvGnYZQ5ETyg,sEwBpFLYsEPu3aUVkh5gwQ,1,87.0,0.0,0.0,0.011494,0.0,0.0
1,-1VbC_BSr9ai4drsuCCJxA,DcBLYSvOuWcNReolRVr12A,0,NaN,0.0,0.0,0.000000,0.0,0.0
2,-6DoXmdXEy_P5N-QZzntgA,Ifw5wqcChnL4zBigtR7NKA,0,NaN,0.0,0.0,0.000000,0.0,0.0
3,-6cPEM5jtb6A2M6mggn47g,lK-YaoQ670dGJrmaqyJTBA,0,NaN,0.0,0.0,0.000000,0.0,0.0
4,-8lbUNlXVSoXqaRRiHiSNg,YiSr-W5BBX6uXnhWPxk82Q,0,NaN,0.0,0.0,0.000000,0.0,0.0


In [36]:
mobility_one = mobility.drop_duplicates(subset=["user_id"]).copy()

# remove old mobility_label first if it already exists
if "mobility_label" in loo_results.columns:
    loo_results = loo_results.drop(columns=["mobility_label"])

# also remove any old suffixed versions if they exist
for c in ["mobility_label_x", "mobility_label_y"]:
    if c in loo_results.columns:
        loo_results = loo_results.drop(columns=[c])

loo_results_with_label = loo_results.merge(
    mobility_one[["user_id", "mobility_label"]],
    on="user_id",
    how="left"
)

print("rows:", len(loo_results_with_label))
print("unique users:", loo_results_with_label["user_id"].nunique())
print("missing labels:", loo_results_with_label["mobility_label"].isna().sum())
print(loo_results_with_label.columns.tolist())

rows: 1159
unique users: 1159
missing labels: 0
['user_id', 'test_business_id', 'found', 'rank', 'recall@10', 'recall@20', 'mrr', 'ndcg@10', 'ndcg@20', 'mobility_label']


In [37]:
summary_metrics = {
    "n_users_evaluated": int(len(loo_results_with_label)),
    "hit_rate": float(loo_results_with_label["found"].mean()),
    "Recall@10": float(loo_results_with_label["recall@10"].mean()),
    "Recall@20": float(loo_results_with_label["recall@20"].mean()),
    "MRR": float(loo_results_with_label["mrr"].mean()),
    "NDCG@10": float(loo_results_with_label["ndcg@10"].mean()),
    "NDCG@20": float(loo_results_with_label["ndcg@20"].mean()),
}

summary_metrics

{'n_users_evaluated': 1159,
 'hit_rate': 0.11993097497842968,
 'Recall@10': 0.01984469370146678,
 'Recall@20': 0.03451251078515962,
 'MRR': 0.010322681069556278,
 'NDCG@10': 0.010343926938994401,
 'NDCG@20': 0.013920528062901392}

In [38]:
group_metrics = loo_results_with_label.groupby("mobility_label")[
    ["found", "recall@10", "recall@20", "mrr", "ndcg@10", "ndcg@20"]
].mean()

group_metrics

,found,recall@10,recall@20,mrr,ndcg@10,ndcg@20
mobility_label,,,,,,
explorer,0.206751,0.042194,0.071730,0.015762,0.018278,0.025406
one_area,0.218750,0.035156,0.046875,0.019448,0.019735,0.022653
sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477
two_area,0.135802,0.012346,0.024691,0.005400,0.003569,0.006475


In [34]:
mobility_one = mobility.drop_duplicates(subset=["user_id"]).copy()

# remove old mobility_label first if it already exists
if "mobility_label" in loo_results.columns:
    loo_results = loo_results.drop(columns=["mobility_label"])

# also remove any old suffixed versions if they exist
for c in ["mobility_label_x", "mobility_label_y"]:
    if c in loo_results.columns:
        loo_results = loo_results.drop(columns=[c])

loo_results_with_label = loo_results.merge(
    mobility_one[["user_id", "mobility_label"]],
    on="user_id",
    how="left"
)

print("rows:", len(loo_results_with_label))
print("unique users:", loo_results_with_label["user_id"].nunique())
print("missing labels:", loo_results_with_label["mobility_label"].isna().sum())
print(loo_results_with_label.columns.tolist())

rows: 1159
unique users: 1159
missing labels: 0
['user_id', 'test_business_id', 'found', 'rank', 'recall@10', 'recall@20', 'mrr', 'ndcg@10', 'ndcg@20', 'mobility_label']


# Starting with global popularity baseline

In [39]:
def retrieve_global_popularity(uid, businesses_eval, train_user_visited, top_n=100):
    visited = train_user_visited.get(uid, set())

    cands = businesses_eval[~businesses_eval["business_id"].isin(visited)].copy()
    cands = cands.sort_values("visit_count", ascending=False).head(top_n).copy()

    cands["retrieval_mode"] = "global_popularity"
    cands["distance_km"] = np.nan
    cands["radius_km"] = np.nan
    cands["score"] = np.log1p(cands["visit_count"])

    return cands

In [40]:
def evaluate_one_user_global(uid, test_business_id, top_n=100):
    cands = retrieve_global_popularity(
        uid=uid,
        businesses_eval=businesses_eval,
        train_user_visited=train_user_visited,
        top_n=top_n
    ).copy()

    if len(cands) == 0:
        return {
            "user_id": uid,
            "test_business_id": test_business_id,
            "found": 0,
            "rank": None,
            "recall@10": 0.0,
            "recall@20": 0.0,
            "mrr": 0.0,
            "ndcg@10": 0.0,
            "ndcg@20": 0.0,
        }

    cands = cands.reset_index(drop=True)
    cands["rank"] = np.arange(1, len(cands) + 1)

    match = cands[cands["business_id"] == test_business_id]

    if len(match) == 0:
        rank = None
        found = 0
    else:
        rank = int(match["rank"].iloc[0])
        found = 1

    return {
        "user_id": uid,
        "test_business_id": test_business_id,
        "found": found,
        "rank": rank,
        "recall@10": recall_at_k(rank, 10),
        "recall@20": recall_at_k(rank, 20),
        "mrr": mrr(rank),
        "ndcg@10": ndcg_at_k(rank, 10),
        "ndcg@20": ndcg_at_k(rank, 20),
    }

In [41]:
global_rows = []

for _, row in loo_eval_filtered.iterrows():
    out = evaluate_one_user_global(
        uid=row["user_id"],
        test_business_id=row["test_business_id"],
        top_n=100
    )
    global_rows.append(out)

loo_results_global = pd.DataFrame(global_rows)

loo_results_global = loo_results_global.merge(
    mobility_one[["user_id", "mobility_label"]],
    on="user_id",
    how="left"
)

print(loo_results_global.shape)
display(loo_results_global.head())

(1159, 10)


,user_id,test_business_id,found,rank,recall@10,recall@20,mrr,ndcg@10,ndcg@20,mobility_label
0,-0MIp6WKJ8QvGnYZQ5ETyg,sEwBpFLYsEPu3aUVkh5gwQ,0,NaN,0.0,0.0,0.0,0.0,0.0,explorer
1,-1VbC_BSr9ai4drsuCCJxA,DcBLYSvOuWcNReolRVr12A,0,NaN,0.0,0.0,0.0,0.0,0.0,sparse
2,-6DoXmdXEy_P5N-QZzntgA,Ifw5wqcChnL4zBigtR7NKA,0,NaN,0.0,0.0,0.0,0.0,0.0,explorer
3,-6cPEM5jtb6A2M6mggn47g,lK-YaoQ670dGJrmaqyJTBA,0,NaN,0.0,0.0,0.0,0.0,0.0,one_area
4,-8lbUNlXVSoXqaRRiHiSNg,YiSr-W5BBX6uXnhWPxk82Q,0,NaN,0.0,0.0,0.0,0.0,0.0,two_area


In [42]:
summary_global = {
    "n_users_evaluated": int(len(loo_results_global)),
    "hit_rate": float(loo_results_global["found"].mean()),
    "Recall@10": float(loo_results_global["recall@10"].mean()),
    "Recall@20": float(loo_results_global["recall@20"].mean()),
    "MRR": float(loo_results_global["mrr"].mean()),
    "NDCG@10": float(loo_results_global["ndcg@10"].mean()),
    "NDCG@20": float(loo_results_global["ndcg@20"].mean()),
}

summary_global

{'n_users_evaluated': 1159,
 'hit_rate': 0.0362381363244176,
 'Recall@10': 0.006902502157031924,
 'Recall@20': 0.01725625539257981,
 'MRR': 0.004279547893875988,
 'NDCG@10': 0.003991152579236819,
 'NDCG@20': 0.0065636586336490724}

In [43]:
group_global = loo_results_global.groupby("mobility_label")[
    ["found", "recall@10", "recall@20", "mrr", "ndcg@10", "ndcg@20"]
].mean()

group_global

,found,recall@10,recall@20,mrr,ndcg@10,ndcg@20
mobility_label,,,,,,
explorer,0.050633,0.021097,0.025316,0.007387,0.009748,0.010828
one_area,0.023438,0.000000,0.015625,0.001196,0.000000,0.003802
sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477
two_area,0.012346,0.000000,0.012346,0.001122,0.000000,0.003444


In [44]:
comparison_overall = pd.DataFrame([
    {
        "model": "global_popularity",
        **summary_global
    },
    {
        "model": "mobility_aware",
        **summary_metrics
    }
])

comparison_overall

,model,n_users_evaluated,hit_rate,Recall@10,Recall@20,MRR,NDCG@10,NDCG@20
0,global_popularity,1159,0.036238,0.006903,0.017256,0.004280,0.003991,0.006564
1,mobility_aware,1159,0.119931,0.019845,0.034513,0.010323,0.010344,0.013921


In [45]:
group_global = group_global.reset_index()
group_global["model"] = "global_popularity"

group_mob = group_metrics.reset_index()
group_mob["model"] = "mobility_aware"

comparison_by_group = pd.concat([group_global, group_mob], ignore_index=True)
comparison_by_group

,mobility_label,found,recall@10,recall@20,mrr,ndcg@10,ndcg@20,model
0,explorer,0.050633,0.021097,0.025316,0.007387,0.009748,0.010828,global_popularity
1,one_area,0.023438,0.000000,0.015625,0.001196,0.000000,0.003802,global_popularity
2,sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477,global_popularity
3,two_area,0.012346,0.000000,0.012346,0.001122,0.000000,0.003444,global_popularity
4,explorer,0.206751,0.042194,0.071730,0.015762,0.018278,0.025406,mobility_aware
5,one_area,0.218750,0.035156,0.046875,0.019448,0.019735,0.022653,mobility_aware
6,sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477,mobility_aware
7,two_area,0.135802,0.012346,0.024691,0.005400,0.003569,0.006475,mobility_aware


# Saving the artifacts

In [1]:
from pathlib import Path
import json

STEP4_DIR = Path(r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step4_outputs")
STEP4_DIR.mkdir(parents=True, exist_ok=True)

print("Saving to:", STEP4_DIR)

Saving to: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step4_outputs


In [2]:
import pandas as pd
from pathlib import Path

# Paths
VISITS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step1_outputs\user_visit_events.parquet"
MOBILITY_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step2_outputs\user_mobility_table.csv"
STEP4_DIR = Path(r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step4_outputs")
STEP4_DIR.mkdir(parents=True, exist_ok=True)

# Load only what is needed
visits = pd.read_csv(VISITS_PATH)
mobility = pd.read_csv(MOBILITY_PATH)

# Time column
TIME_COL = "event_time"
visits[TIME_COL] = pd.to_datetime(visits[TIME_COL], errors="coerce")

# Build leave-one-out split exactly like your Step 4
visits_loo = visits.dropna(subset=[TIME_COL]).copy()
visits_loo = visits_loo.sort_values(["user_id", TIME_COL])

loo_rows = []
train_rows = []

for uid, dfu in visits_loo.groupby("user_id"):
    if len(dfu) < 5:
        continue

    dfu = dfu.sort_values(TIME_COL)
    train = dfu.iloc[:-1].copy()
    test = dfu.iloc[-1].copy()

    loo_rows.append({
        "user_id": uid,
        "test_business_id": test["business_id"],
        "test_time": test[TIME_COL],
        "n_train_events": len(train),
        "n_total_events": len(dfu),
    })

    train_rows.append(train)

loo_eval = pd.DataFrame(loo_rows)
train_visits = pd.concat(train_rows, ignore_index=True)

# Filter to the profiled users, same as your Step 4
profiled_users = set(mobility["user_id"])

loo_eval_filtered = loo_eval[loo_eval["user_id"].isin(profiled_users)].copy()
train_visits_filtered = train_visits[train_visits["user_id"].isin(profiled_users)].copy()

# Export LightFM files
train_interactions = (
    train_visits_filtered[["user_id", "business_id"]]
    .drop_duplicates()
    .copy()
)
train_interactions.to_csv(STEP4_DIR / "train_interactions.csv", index=False)

test_interactions = (
    loo_eval_filtered[["user_id", "test_business_id"]]
    .rename(columns={"test_business_id": "business_id"})
    .copy()
)
test_interactions.to_csv(STEP4_DIR / "test_interactions.csv", index=False)

print("Saved:", STEP4_DIR / "train_interactions.csv")
print("Saved:", STEP4_DIR / "test_interactions.csv")
print()
print("loo_eval_filtered users:", loo_eval_filtered["user_id"].nunique())
print("train_visits_filtered users:", train_visits_filtered["user_id"].nunique())
print("train_interactions shape:", train_interactions.shape)
print("test_interactions shape:", test_interactions.shape)

display(train_interactions.head())
display(test_interactions.head())

Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step4_outputs\train_interactions.csv
Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step4_outputs\test_interactions.csv

loo_eval_filtered users: 1159
train_visits_filtered users: 1159
train_interactions shape: (14412, 2)
test_interactions shape: (1159, 2)


,user_id,business_id
0,-0MIp6WKJ8QvGnYZQ5ETyg,UTgUhbpM5hRVDB8Nu7-lpw
1,-0MIp6WKJ8QvGnYZQ5ETyg,jwpZzqoPFLuHE-FwLp41cQ
2,-0MIp6WKJ8QvGnYZQ5ETyg,uXziexv0fsfRMw1hEdifow
3,-0MIp6WKJ8QvGnYZQ5ETyg,E7pGpkLw7i73oeuf8tvJog
4,-0MIp6WKJ8QvGnYZQ5ETyg,1wKbk-FtJBBidd1k8s09DA


,user_id,business_id
0,-0MIp6WKJ8QvGnYZQ5ETyg,sEwBpFLYsEPu3aUVkh5gwQ
1,-1VbC_BSr9ai4drsuCCJxA,DcBLYSvOuWcNReolRVr12A
2,-6DoXmdXEy_P5N-QZzntgA,Ifw5wqcChnL4zBigtR7NKA
3,-6cPEM5jtb6A2M6mggn47g,lK-YaoQ670dGJrmaqyJTBA
4,-8lbUNlXVSoXqaRRiHiSNg,YiSr-W5BBX6uXnhWPxk82Q


In [ ]:
# LightFM training interactions
train_interactions = train_visits_filtered[["user_id", "business_id"]].copy()
train_interactions.to_csv(STEP4_DIR / "train_interactions.csv", index=False)

print("Saved train_interactions.csv")
print(train_interactions.shape)
display(train_interactions.head())

In [ ]:
# LightFM test interactions (held-out last item per user)
test_interactions = loo_eval_filtered[["user_id", "test_business_id"]].copy()
test_interactions = test_interactions.rename(columns={"test_business_id": "business_id"})
test_interactions.to_csv(STEP4_DIR / "test_interactions.csv", index=False)

print("Saved test_interactions.csv")
print(test_interactions.shape)
display(test_interactions.head())

In [3]:
train_interactions_full = train_visits_filtered.copy()
train_interactions_full.to_csv(STEP4_DIR / "train_interactions_full.csv", index=False)

print("Saved train_interactions_full.csv")
print(train_interactions_full.shape)
display(train_interactions_full.head())

Saved train_interactions_full.csv
(16579, 14)


,user_id,business_id,event_time,event_type,stars_x,name,latitude,longitude,city,state,categories,stars_y,review_count,is_open
0,-0MIp6WKJ8QvGnYZQ5ETyg,UTgUhbpM5hRVDB8Nu7-lpw,2018-06-11 22:57:47+00:00,review,4.0,China Cafe,39.610539,-75.074803,Franklinville,NJ,"Chinese, Restaurants",4.0,18,1
1,-0MIp6WKJ8QvGnYZQ5ETyg,jwpZzqoPFLuHE-FwLp41cQ,2020-03-15 00:33:43+00:00,review,4.0,Bistro Di Marino,39.918191,-75.075328,Collingswood,NJ,"Italian, Salad, Restaurants, Seafood",4.0,285,1
2,-0MIp6WKJ8QvGnYZQ5ETyg,uXziexv0fsfRMw1hEdifow,2020-04-11 22:26:07+00:00,review,4.0,Pasta Pomodoro,39.750364,-75.075305,Sewell,NJ,"Restaurants, Italian, Seafood, Sandwiches",4.0,68,1
3,-0MIp6WKJ8QvGnYZQ5ETyg,E7pGpkLw7i73oeuf8tvJog,2020-04-25 23:28:16+00:00,review,4.0,Cinder Bar,39.723962,-75.017087,Williamstown,NJ,"Bars, Nightlife, Restaurants, American (New), ...",4.0,203,1
4,-0MIp6WKJ8QvGnYZQ5ETyg,1wKbk-FtJBBidd1k8s09DA,2020-05-31 00:19:27+00:00,review,1.0,Dia De Los Burritos,39.731998,-75.129396,Pitman,NJ,"Mexican, Restaurants, Tex-Mex",4.0,114,1


In [4]:
train_interactions_dedup = (
    train_visits_filtered[["user_id", "business_id"]]
    .drop_duplicates()
    .copy()
)
train_interactions_dedup.to_csv(STEP4_DIR / "train_interactions_dedup.csv", index=False)

print("Saved train_interactions_dedup.csv")
print(train_interactions_dedup.shape)
display(train_interactions_dedup.head())

Saved train_interactions_dedup.csv
(14412, 2)


,user_id,business_id
0,-0MIp6WKJ8QvGnYZQ5ETyg,UTgUhbpM5hRVDB8Nu7-lpw
1,-0MIp6WKJ8QvGnYZQ5ETyg,jwpZzqoPFLuHE-FwLp41cQ
2,-0MIp6WKJ8QvGnYZQ5ETyg,uXziexv0fsfRMw1hEdifow
3,-0MIp6WKJ8QvGnYZQ5ETyg,E7pGpkLw7i73oeuf8tvJog
4,-0MIp6WKJ8QvGnYZQ5ETyg,1wKbk-FtJBBidd1k8s09DA


In [50]:
loo_eval_filtered.to_csv(STEP4_DIR / "loo_eval_filtered.csv", index=False)
print("Saved loo_eval_filtered.csv")

Saved loo_eval_filtered.csv


In [51]:
loo_results_global.to_csv(STEP4_DIR / "loo_results_global.csv", index=False)
group_global.to_csv(STEP4_DIR / "loo_group_metrics_global.csv")

with open(STEP4_DIR / "summary_global.json", "w", encoding="utf-8") as f:
    json.dump(summary_global, f, indent=2)

print("Saved global popularity artifacts")

Saved global popularity artifacts


In [52]:
loo_results_with_label.to_csv(STEP4_DIR / "loo_results_mobility_aware.csv", index=False)
group_metrics.to_csv(STEP4_DIR / "loo_group_metrics_mobility_aware.csv")

with open(STEP4_DIR / "summary_mobility_aware.json", "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=2)

print("Saved mobility-aware artifacts")

Saved mobility-aware artifacts


In [53]:
print("\n=== comparison_overall ===")
display(comparison_overall)

print("\n=== comparison_by_group ===")
display(comparison_by_group)


=== comparison_overall ===


,model,n_users_evaluated,hit_rate,Recall@10,Recall@20,MRR,NDCG@10,NDCG@20
0,global_popularity,1159,0.036238,0.006903,0.017256,0.004280,0.003991,0.006564
1,mobility_aware,1159,0.119931,0.019845,0.034513,0.010323,0.010344,0.013921



=== comparison_by_group ===


,mobility_label,found,recall@10,recall@20,mrr,ndcg@10,ndcg@20,model
0,explorer,0.050633,0.021097,0.025316,0.007387,0.009748,0.010828,global_popularity
1,one_area,0.023438,0.000000,0.015625,0.001196,0.000000,0.003802,global_popularity
2,sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477,global_popularity
3,two_area,0.012346,0.000000,0.012346,0.001122,0.000000,0.003444,global_popularity
4,explorer,0.206751,0.042194,0.071730,0.015762,0.018278,0.025406,mobility_aware
5,one_area,0.218750,0.035156,0.046875,0.019448,0.019735,0.022653,mobility_aware
6,sparse,0.039316,0.005128,0.015385,0.004807,0.003958,0.006477,mobility_aware
7,two_area,0.135802,0.012346,0.024691,0.005400,0.003569,0.006475,mobility_aware


In [54]:
step4_meta = {
    "evaluation_type": "leave_one_out",
    "evaluated_users": int(len(loo_results_with_label)),
    "candidate_budget": 100,
    "models": ["global_popularity", "mobility_aware"],
    "notes": "Mobility-aware retriever uses Step 2 mobility labels and Step 3 hub-aware gating with personalized radius."
}

with open(STEP4_DIR / "step4_meta.json", "w", encoding="utf-8") as f:
    json.dump(step4_meta, f, indent=2)

print("Saved step4_meta.json")

Saved step4_meta.json
